# CMR-RIS: CardioMetabolic-Renal Risk Intelligence System
### End-to-End Research Pipeline (Phases 0 through 14)

This notebook executes and verifies all phases of the CMR-RIS project:
- **Phase 0-1**: Environment Setup & Dataset Validation
- **Phase 2-4**: Leakage-Free Preprocessing & Medical Feature Engineering
- **Phase 5-7**: Independent vs Multi-Task Modeling & Screening Champion Selection
- **Phase 8**: Composite CardioMetabolic-Renal Index (CMRI)
- **Phase 9**: Conformal Prediction Uncertainty Quantification (90% CI)
- **Phase 10**: Longitudinal Trajectory Engine (Theil-Sen + Mann-Kendall)
- **Phase 11**: Explainable AI (SHAP attributions + Clinical Counterfactuals)
- **Phase 12**: Auditable MCDA Screening Priority Engine
- **Phase 13**: Subgroup Fairness Audit across Demographics
- **Phase 14**: Interactive Streamlit Dashboard Deployment (`app.py`)

In [ ]:
# Cell 1: Environment & Module Setup
import sys
import os

# Ensure current working directory is the project root (whether run from root or notebooks/)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Environment initialized successfully! Project Root: {os.getcwd()}")

In [ ]:
# Cell 2: Phase 2 & 4 — Preprocessing Pipeline Verification
from src.preprocessing import load_diabetes, load_heart, load_ckd, load_framingham, load_shared_continuum_data

print("Testing Preprocessing Pipelines (Zero Data Leakage)...")
for name, loader in [('Diabetes', load_diabetes), ('Heart', load_heart), ('CKD', load_ckd), ('Framingham', load_framingham)]:
    X_tr, X_te, y_tr, y_te, scaler, pipe = loader(engineered=True)
    print(f"  [PASS] {name:12s} | Train: {X_tr.shape} | Test: {X_te.shape} | Engineered Features: {len(X_tr.columns)}")

shared = load_shared_continuum_data()
for k, v in shared.items():
    print(f"  [PASS] Shared Continuum {k:8s} | Train: {v[0].shape} | Test: {v[1].shape}")

In [ ]:
# Cell 3: Phase 5, 6 & 7 — Model Benchmark & Champion Selection
from src.modeling import run_full_modeling_suite

df_all, champions = run_full_modeling_suite()
print("\nModel Comparison Summary:")
display_cols = ['Disease', 'Model_Type', 'Algorithm', 'Test_Recall', 'Test_ROC_AUC', 'Test_F1', 'Test_Accuracy']
print(df_all[display_cols].to_string(index=False))

print("\nSelected Champions (Recall Prioritized):")
for d, info in champions.items():
    print(f"  ★ {d.upper()}: {info['Algorithm']} | Recall: {info['Test_Recall']:.3f} | ROC-AUC: {info['Test_ROC_AUC']:.3f}")

In [ ]:
# Cell 4: Phase 8 — Composite CardioMetabolic-Renal Index (CMRI)
from src.cmri import CMRIndexCalculator

cmri_calc = CMRIndexCalculator()
test_cases = [
    ('Healthy Individual', 0.05, 0.08, 0.02),
    ('Isolated Diabetes', 0.85, 0.15, 0.10),
    ('Isolated Cardiac', 0.10, 0.90, 0.12),
    ('Diabetic Nephropathy', 0.85, 0.20, 0.88),
    ('Triple Comorbidity', 0.90, 0.85, 0.92)
]

for name, pd_v, ph_v, pc_v in test_cases:
    res = cmri_calc.calculate(pd_v, ph_v, pc_v)
    print(f"[{res['risk_category']:20s}] {name:25s} -> CMRI: {res['cmri_percentage']:5.1f}% | Audit: {res['formula_audit']}")

In [ ]:
# Cell 5: Phase 9 — Split Conformal Prediction Uncertainty Calibration
from src.uncertainty import calibrate_all_champions

calibrator, df_calib = calibrate_all_champions(confidence_level=0.90)
print("Conformal Uncertainty Calibration:")
print(df_calib.to_string(index=False))

print("\nCalibrated Risk Intervals for Example Patient:")
for d, p in [('diabetes', 0.78), ('heart', 0.42), ('ckd', 0.15)]:
    res = calibrator.predict_interval(d, p)
    print(f"  {d.capitalize():10s}: {res['formatted']}")

In [ ]:
# Cell 6: Phase 10 — Longitudinal Trajectory Engine
from src.trajectory import classify_trajectory, simulate_patient_trajectory

scenarios = [
    ('Stable Patient', [0, 1, 2, 3], [0.22, 0.23, 0.22, 0.24]),
    ('Improving Lifestyle', [0, 1, 2, 3], [0.65, 0.58, 0.49, 0.42]),
    ('Gradual Metabolic Drift', [0, 1, 2, 3], [0.30, 0.35, 0.38, 0.44]),
    ('Acute Deterioration', [0, 1, 2, 3], [0.25, 0.38, 0.55, 0.72])
]

for name, ts, rs in scenarios:
    res = classify_trajectory(ts, rs)
    print(f"[{res['trajectory_status']:22s}] {name:30s} | Slope: {res['annual_drift_percentage']:+5.1f}%/yr | MK p: {res['mann_kendall']['p_value']:.4f}")

In [ ]:
# Cell 7: Phase 11 — Explainable AI (SHAP & Counterfactuals)
from src.explainability import ModelExplainer

heart_champ = joblib.load('models/heart_champion.pkl')
X_tr_h, X_te_h, _, _, _, _ = load_heart(engineered=True)
explainer = ModelExplainer(heart_champ, background_data=X_tr_h)
sample_pt = X_te_h.iloc[0:1]

top_drivers = explainer.explain_patient(sample_pt, top_k=5)
print("Top Local Risk Drivers (SHAP):")
print(top_drivers[['Feature', 'Standardized_Value', 'SHAP_Impact', 'Direction']].to_string(index=False))

cf = explainer.compute_counterfactual(sample_pt, {'trestbps': -0.8, 'chol': -0.5})
print(f"\nActionable Counterfactual Simulation:\n{cf['clinical_narrative']}")

In [ ]:
# Cell 8: Phase 12 & 13 — MCDA Priority Triage & Subgroup Fairness Audit
from src.priority import MCDAPriorityEngine
from src.fairness import run_full_fairness_audit

engine = MCDAPriorityEngine()
prio_res = engine.compute_priority(0.78, 0.085, {'diabetes': 0.85, 'heart': 0.70, 'ckd': 0.90})
print(f"Triage Priority Tier: {prio_res['priority_tier']} ({prio_res['priority_percentage']:.1f}%)")
print(f"Formula Audit: {prio_res['formula_audit']}")

print("\nRunning Demographic Fairness Audit:")
df_fair = run_full_fairness_audit()
print(df_fair[['Disease', 'Subgroup_Dimension', 'Subgroup', 'Sample_N', 'Recall', 'F1_Score', 'Accuracy']].to_string(index=False))

## Phase 14: Streamlit Dashboard Deployment
To launch the interactive dashboard, run in your terminal:
```bash
streamlit run app.py
```